# 🎭 KoCulture-Dialogues SLM 파인튜닝 (QLoRA)

**한국 신조어 데이터셋으로 Qwen2.5-3B 모델을 파인튜닝합니다.**

## 실행 환경
- **권장**: Google Colab + T4 GPU (무료) 또는 A100 (Pro)
- **소요 시간**: T4에서 약 4시간 5분 시간 
- **VRAM**: 6.07 GB 사용

## 진행 순서
1. GPU 확인 → 라이브러리 설치
2. 데이터셋 로드 및 전처리
3. 모델 로드 (4-bit 양자화)
4. **파인튜닝 전** 답변 테스트 (비교용)
5. LoRA 설정 → 학습
6. **파인튜닝 후** 답변 테스트
7. 어댑터 저장

> ⚠️ **시작 전 필수**: Colab에서 `런타임 > 런타임 유형 변경 > GPU > T4` 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 0. GPU 환경 확인

먼저 GPU가 제대로 잡혔는지 확인합니다.

In [ ]:
!nvidia-smi

## Step 1. 필수 라이브러리 설치

QLoRA 학습에 필요한 라이브러리들을 설치합니다. 버전 고정이 중요해요 — 안 그러면 호환성 문제가 자주 발생합니다.

In [ ]:
!pip install -q -U \
    transformers==4.46.0 \
    peft==0.13.2 \
    bitsandbytes==0.45.3 \
    trl==0.11.4 \
    datasets==3.0.0 \
    accelerate==1.0.1 \

print("✅ 설치 완료. 런타임 재시작이 필요할 수 있어요.")

## Step 2. 라이브러리 import

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Step 3. 데이터셋 로드 & 살펴보기

In [ ]:
from datasets.builder import VerificationMode
# KoCulture-Dialogues 로드 (10,356 행)
ds = load_dataset("huggingface-KREW/KoCulture-Dialogues", split="train", verification_mode=VerificationMode.NO_CHECKS)

print(f"전체 데이터 수: {len(ds)}")
print(f"필드: {ds.column_names}")
print()
print("=== 샘플 3개 ===")
for i in [0, 100, 5000]:
    print(f"\n[{ds[i]['title']}]")
    print(f"Q: {ds[i]['question']}")
    print(f"A: {ds[i]['answer']}")

## Step 4. 데이터 전처리

원본은 `title/question/answer` 구조지만, 학습엔 대화 형식이 필요합니다.
`question`을 user 메시지로, `answer`를 assistant 메시지로 변환합니다.

In [ ]:
def to_chat_format(example):
    return {
        "messages": [
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": example["answer"]},
        ]
    }

ds = ds.map(to_chat_format, remove_columns=ds.column_names)

# Train/Validation 9:1 분할
ds = ds.train_test_split(test_size=0.1, seed=42)

print(f"Train: {len(ds['train'])}")
print(f"Eval:  {len(ds['test'])}")
print()
print("=== 변환된 샘플 ===")
print(ds["train"][0])

## Step 5. 모델 & 토크나이저 로드 (4-bit 양자화)

Qwen2.5-3B-Instruct를 4-bit로 압축하여 메모리를 절약합니다.
원래 약 6GB짜리 모델이 약 2GB로 줄어듭니다.

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import prepare_model_for_kbit_training
import torch

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# 4-bit 양자화 설정 (QLoRA의 'Q' 부분)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # Normal Float 4-bit
    bnb_4bit_compute_dtype=torch.float16, # T4 GPU는 float16에 더 최적화되어 있음
    bnb_4bit_use_double_quant=True,     # 양자화 상수도 양자화
)

# 토크나이저
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 모델 로드 (자동으로 GPU에 올림)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16, # T4 GPU는 float16에 더 최적화되어 있음
)

# LoRA 학습 준비 (gradient checkpointing 등)
model = prepare_model_for_kbit_training(model)

print(f"✅ 모델 로드 완료")
print(f"메모리 사용량: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 6. ⭐ 파인튜닝 **전** 답변 테스트

학습 시작 전에 베이스 모델이 신조어를 어떻게 다루는지 미리 보고 갑니다.
나중에 비교용 자료로 쓸 수 있습니다.

In [ ]:
def chat(prompt, max_new_tokens=200):
    """모델에게 질문을 던지고 답변을 받습니다."""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

# 평가용 질문 (학습 데이터에 없는 신조어 위주로!)
test_questions = [
    "친구가 게임에서 봉산탈춤 추고 있다는데 뭔 뜻이야?",
    "내 추구미는 미니멀한 인테리어인데 어떻게 꾸미면 좋을까?",
    "어제 콘서트 진짜 어마무시했어",
    "쟤 음주운전하다 경찰서 정모 갔대",
    "오늘 발표 폼 미쳤다",
]

print("=" * 70)
print("🎯 파인튜닝 BEFORE")
print("=" * 70)
before_results = {}
for q in test_questions:
    answer = chat(q)
    before_results[q] = answer
    print(f"\nQ: {q}")
    print(f"A: {answer}")
    print("-" * 70)

## Step 7. LoRA 설정

학습할 LoRA 어댑터를 정의합니다. 핵심 하이퍼파라미터:
- `r=16`: LoRA 랭크. 클수록 표현력↑, 메모리↑
- `lora_alpha=32`: 스케일링 계수 (보통 r의 2배)
- `target_modules`: LoRA를 적용할 레이어들

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print("✅ LoRA 설정 완료")

## Step 8. 학습 설정

SFTConfig로 학습 하이퍼파라미터를 정의합니다. T4에서는 batch_size=2 권장합니다.

In [ ]:
sft_config = SFTConfig(
    # 출력 경로
    output_dir="/content/drive/MyDrive/koculture/checkpoints",  # ← 변경: 드라이브로

    # 학습 길이
    num_train_epochs=3,
    per_device_train_batch_size=2,       # T4 기준. A100이면 8까지 OK
    gradient_accumulation_steps=4,        # 실효 배치 = 2 × 4 = 8

    # 학습률
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # 메모리 최적화
    bf16=False,                          # ← 변경: T4는 bf16 미지원
    fp16=True,                           # ← 추가: T4에선 fp16 사용
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,

    # 로깅 & 저장
    logging_steps=20,
    save_strategy="steps",               # ← 변경: epoch → steps
    save_steps=200,                      # ← 추가: 약 30분마다 저장
    eval_strategy="epoch",
    save_total_limit=2,

    # SFT 전용
    max_seq_length=512,
    packing=False,

    # 기타
    report_to="none",                    # wandb 등 비활성화
    seed=42,
)

print("✅ 학습 설정 완료")

## Step 9. 🚀 학습 실행!

SFTTrainer가 LoRA 적용 + 학습을 자동으로 처리합니다.
진행 상황은 loss로 확인할 수 있습니다.

In [ ]:
import os
from transformers.trainer_utils import get_last_checkpoint

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    peft_config=lora_config,
    tokenizer=tokenizer,
)

# 드라이브에 이전 체크포인트가 있는지 확인
ckpt_dir = sft_config.output_dir
last_ckpt = get_last_checkpoint(ckpt_dir) if os.path.isdir(ckpt_dir) else None

if last_ckpt:
    print(f"🔄 체크포인트 발견 → 여기서 이어서 학습: {last_ckpt}")
    trainer.train(resume_from_checkpoint=last_ckpt)
else:
    print("🆕 처음부터 학습 시작")
    trainer.train()

print("\n✅ 이번 세션 학습 구간 완료!")

## Step 10. 어댑터 저장

학습된 LoRA 어댑터만 저장합니다 (~120MB).

In [ ]:
SAVE_PATH = "./output/koculture-lora-final"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# Colab에서 Google Drive에 백업하고 싶다면:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r ./output/koculture-lora-final /content/drive/MyDrive/

print(f"✅ 저장 완료: {SAVE_PATH}")
!du -sh {SAVE_PATH}

## Step 10.5. 🤗 Hugging Face Hub에 어댑터 업로드

학습된 LoRA 어댑터(~120MB)는 깃허브에 직접 올리기엔 크기 때문에(50MB 경고/100MB 거부),
모델 전용 호스팅인 Hugging Face Hub에 올립니다. 업로드 후 README에는 링크만 걸면 됩니다.

> 사전 준비: huggingface.co 가입 → Settings → Access Tokens → **write 권한** 토큰 발급

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # 실행하면 입력창이 떠. write 토큰 붙여넣기

In [ ]:
from huggingface_hub import whoami
print(whoami()["name"])

In [ ]:
REPO_ID = "DdingDDing0103/koculture-qwen2.5-3b-lora"

trainer.model.push_to_hub(REPO_ID, commit_message="Add KoCulture QLoRA adapter")  # ← trainer.model
tokenizer.push_to_hub(REPO_ID, commit_message="Add tokenizer")

print(f"✅ 업로드 완료: https://huggingface.co/{REPO_ID}")

## Step 11. ⭐ 파인튜닝 **후** 답변 테스트

같은 질문을 학습된 모델에게 던져봅니다.

In [ ]:
print("=" * 70)
print("🎯 파인튜닝 AFTER")
print("=" * 70)

after_results = {}
for q in test_questions:
    answer = chat(q)
    after_results[q] = answer
    print(f"\nQ: {q}")
    print(f"A: {answer}")
    print("-" * 70)

## Step 12. Before vs After 비교 표

블로그에 그대로 넣을 수 있는 비교 표를 만듭니다.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "질문": test_questions,
    "파인튜닝 전": [before_results[q] for q in test_questions],
    "파인튜닝 후": [after_results[q] for q in test_questions],
})

# 보기 좋게 출력
pd.set_option("display.max_colwidth", None)
display(comparison)

# CSV로 저장 (블로그용)
comparison.to_csv("./output/before_after_comparison.csv", index=False, encoding="utf-8-sig")
print("\n✅ CSV 저장 완료: ./output/before_after_comparison.csv")

In [ ]:
import torch, os, matplotlib.pyplot as plt

os.makedirs("images", exist_ok=True)
hist = trainer.state.log_history

# 1) Loss 곡선 → images/train_loss.png  (README IV-2)
train = [(h["step"], h["loss"]) for h in hist if "loss" in h]
evals = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(8, 5))
if train: plt.plot(*zip(*train), label="train")
if evals: plt.plot(*zip(*evals), marker="o", label="eval")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Training / Validation Loss")
plt.savefig("images/train_loss.png", dpi=150, bbox_inches="tight"); plt.show()

# 2) 효율성 수치  (README IV-3)
runtime = next((h["train_runtime"] for h in reversed(hist) if "train_runtime" in h), None)
print(f"시작 loss : {train[0][1]:.3f}  →  종료 loss : {train[-1][1]:.3f}")
print(f"학습 시간 : {runtime/60:.1f} 분" if runtime else "학습 시간: (로그에 없음)")
print(f"최대 GPU 메모리 : {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print("어댑터 크기 : 약 120 MB (fp32, HF 업로드 확인됨)")

## Step 13. (참고) 나중에 어댑터만 불러와서 쓰기

학습이 다 끝난 후, 어댑터만 따로 불러와서 추론하는 방법입니다.
블로그의 `inference.py` 데모 코드로 활용 가능.

In [ ]:
# 🔁 새 세션에서는 이 코드만 실행하면 됩니다

# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel
# import torch
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )
#
# base = AutoModelForCausalLM.from_pretrained(
#     "Qwen/Qwen2.5-3B-Instruct",
#     quantization_config=bnb_config,
#     device_map="auto",
# )
# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
# model = PeftModel.from_pretrained(base, "./output/koculture-lora-final")
# model.eval()

print("위 코드는 새 세션에서 어댑터만 불러올 때 사용합니다.")